In [97]:
using LowLevelFEM
import LowLevelFEM as L
using LinearAlgebra

In [98]:
structured_rect_mesh(x0=1, n=200, order=2)
mat = Material("body")
Pu = Problem([mat], type=:VectorField, dim=2, field=:u)

Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 160801, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs)

In [99]:
#export ChainSum, TransposedChain, withgauss, reduced, full

import Base: +, adjoint, transpose, *
import LowLevelFEM: ⋅

In [100]:
"""
    ChainSumTerm(chain, gauss)

Internal storage for one term of a compound operator expression.

`chain` is usually an `OpApplied` or a `MatrixChain`.
`gauss` stores the preferred integration rule for this term.
"""
struct ChainSumTerm
    chain::Any
    gauss::Any
end


ChainSumTerm

In [101]:
"""
    ChainSum(terms)

Symbolic sum of operator chains.

Example:

```julia
B = Grad(Pu) ⋅ A + Pu ⋅ G
```

stores two terms without assembling anything.
"""
struct ChainSum
    terms::Vector{ChainSumTerm}
end


ChainSum

In [102]:
"""
TransposedChain(chain)

Symbolic transpose marker for compound expressions.

This does not assemble or numerically transpose anything immediately.
It is interpreted during expansion of the bilinear form.
"""
struct TransposedChain
    chain::Any
end


TransposedChain

In [103]:
"""
CompoundChain(left, mats)

Temporary object representing

```
left ⋅ M1 ⋅ M2 ⋯
```

before the right-hand side is supplied.
"""
struct CompoundChain
    left::Any
    mats::Vector{Any}
end


CompoundChain

In [104]:
"""
CompoundBilinear(left, mats, right)

Temporary object representing

```
left ⋅ M1 ⋅ M2 ⋯ ⋅ right
```

where `left` and/or `right` may be `ChainSum`.
"""
struct CompoundBilinear
    left::Any
    mats::Vector{Any}
    right::Any
end


CompoundBilinear

In [105]:
struct CompoundBilinearWeight
    CB::CompoundBilinear
    weight::Any
end

In [106]:
*(A::CompoundBilinear, B) = CompoundBilinearWeight(A, B)

* (generic function with 311 methods)

In [107]:
withgauss(chain, gauss) = ChainSumTerm(chain, gauss)
reduced(chain) = withgauss(chain, :reduced)
full(chain) = withgauss(chain, :full)

_as_sumterm(t::ChainSumTerm) = t
_as_sumterm(x) = ChainSumTerm(x, :full)

_chain_sum(xs...) = ChainSum([_as_sumterm(x) for x in xs])

_is_chain_object(x) =
    x isa L.OpApplied ||
    x isa L.MatrixChain ||
    x isa ChainSum ||
    x isa ChainSumTerm ||
    x isa TransposedChain

function _coeff_mats(c)
    if c isa AbstractMatrix
        _check_coeff_matrix(c)
        return Any[c]
    elseif c isa TensorField
        return Any[tensorfield_to_matrix(c)]
    elseif c isa Number || c isa ScalarField
        return Any[c]
    else
        error("Compound operator: invalid coefficient type $(typeof(c)).")
    end
end

function _op_and_mats(x)
    if x isa L.OpApplied
        return x, Any[]
    elseif x isa L.MatrixChain
        return x.a, copy(x.mats)
    else
        error("Compound operator term must be OpApplied or MatrixChain, got $(typeof(x)).")
    end
end

struct _SideTerm
    chain::Any
    transposed::Bool
    gauss::Any
end

function _side_terms(x; transposed=false)
    if x isa TransposedChain
        return _side_terms(x.chain; transposed=(!transposed))
    elseif x isa ChainSum
        return [_SideTerm(t.chain, transposed, t.gauss) for t in x.terms]

    elseif x isa ChainSumTerm
        return [_SideTerm(x.chain, transposed, x.gauss)]

    elseif x isa L.OpApplied || x isa L.MatrixChain
        return [_SideTerm(x, transposed, :full)]

    else
        error("Compound operator: invalid side type $(typeof(x)).")
    end
end

function _maybe_transpose_mats(mats, transposed::Bool)
    if transposed
        return Any[adjoint(M) for M in reverse(mats)]
    else
        return copy(mats)
    end
end

function _select_gauss(left::_SideTerm, right::_SideTerm, gauss)
    gauss !== :auto && return gauss
    if left.gauss == right.gauss
        return left.gauss
    end

    return :full
end

function _compress_mats(mats)
    isempty(mats) && return 1.0

    C = mats[1]
    for i in 2:length(mats)
        C = C * mats[i]
    end

    return C
end

function _make_bilinear_term(left::_SideTerm, middle::Vector{Any}, right::_SideTerm)
    left_op, left_mats = _op_and_mats(left.chain)
    right_op, right_mats = _op_and_mats(right.chain)
    mats = Any[
        #_maybe_transpose_mats(left_mats, left.transposed)...,
        left_mats...,
        middle...,
        #_maybe_transpose_mats(right_mats, right.transposed)...
        adjoint.(reverse(right_mats))...
    ]

    coef = isempty(mats) ? 1.0 : mats

    #coef = _compress_mats(mats)

    return L.BilinearTerm(left_op, coef, right_op)
end

function _coeff_mats(c)
    if c isa AbstractMatrix
        return Any[c]
    elseif c isa Number || c isa ScalarField || c isa TensorField
        return Any[c]
    else
        error("Compound operator: invalid coefficient type $(typeof(c)).")
    end
end

_coeff_mats (generic function with 1 method)

In [108]:
# ---------------------------------------------------------------------------

# Addition: build ChainSum

# ---------------------------------------------------------------------------

+(a::ChainSum, b::ChainSum) =
    ChainSum([a.terms...; b.terms...])

+(a::ChainSum, b::Union{L.OpApplied,L.MatrixChain,ChainSumTerm,TransposedChain}) =
    ChainSum([a.terms...; _as_sumterm(b)])

+(a::Union{L.OpApplied,L.MatrixChain,ChainSumTerm,TransposedChain}, b::ChainSum) =
    ChainSum([_as_sumterm(a); b.terms...])

+(a::Union{L.OpApplied,L.MatrixChain,ChainSumTerm,TransposedChain},
    b::Union{L.OpApplied,L.MatrixChain,ChainSumTerm,TransposedChain}) =
    _chain_sum(a, b)

+ (generic function with 247 methods)

In [109]:
# ---------------------------------------------------------------------------

# Symbolic transpose

# ---------------------------------------------------------------------------

adjoint(x::Union{ChainSum,ChainSumTerm,L.MatrixChain,L.OpApplied}) =
    TransposedChain(x)

transpose(x::Union{ChainSum,ChainSumTerm,L.MatrixChain,L.OpApplied}) =
    TransposedChain(x)

transpose (generic function with 53 methods)

In [110]:
# ---------------------------------------------------------------------------

# Dot products involving ChainSum / TransposedChain

# ---------------------------------------------------------------------------

function ⋅(left::Union{ChainSum,TransposedChain,ChainSumTerm}, c)
    return CompoundChain(left, _coeff_mats(c))
end

function ⋅(cc::CompoundChain, c)
    append!(cc.mats, _coeff_mats(c))
    return cc
end

function ⋅(cc::CompoundChain, right::Union{ChainSum,TransposedChain,ChainSumTerm,L.OpApplied,L.MatrixChain})
    return CompoundBilinear(cc.left, cc.mats, right)
end

function ⋅(left::Union{ChainSum,TransposedChain,ChainSumTerm},
    right::Union{ChainSum,TransposedChain,ChainSumTerm,L.OpApplied,L.MatrixChain})
    return CompoundBilinear(left, Any[], right)
end

function ⋅(left::Union{L.OpApplied,L.MatrixChain},
    right::Union{ChainSum,TransposedChain,ChainSumTerm})
    return CompoundBilinear(left, Any[], right)
end

dot (generic function with 74 methods)

In [111]:
function ⋅(A::AbstractMatrix, op::L.OpApplied)
    return op ⋅ adjoint(A)
end

function ⋅(A::AbstractMatrix, ch::L.MatrixChain)
    return ch ⋅ adjoint(A)
end

function ⋅(A::Union{Number,ScalarField}, op::L.OpApplied)
    return op ⋅ A
end

dot (generic function with 74 methods)

In [112]:
# ---------------------------------------------------------------------------

# Integral wrapper

# ---------------------------------------------------------------------------

function ∫(expr::CompoundBilinear;
    Ω=nothing, Γ=nothing, weight=nothing,
    gauss=:auto)
    #@show typeof(expr.left)
    #@show typeof(expr.right)

    left_terms = _side_terms(expr.left)
    right_terms = _side_terms(expr.right)
    #@show left_terms
    #@show right_terms

    K = nothing

    for LL in left_terms
        for R in right_terms
            bt = _make_bilinear_term(LL, expr.mats, R)
            #@show bt
            g = _select_gauss(LL, R, gauss)

            Ki = L.∫(bt; Ω=Ω, Γ=Γ, weight=weight, gauss=g)
            K = K === nothing ? Ki : K + Ki
        end
    end

    return K
end

∫ (generic function with 2 methods)

In [113]:
∫(expr::CompoundBilinearWeight; Ω=nothing, Γ=nothing, gauss=:auto) = ∫(expr.CB; Ω=Ω, Γ=Γ, weight=expr.weight, gauss=gauss)

∫ (generic function with 2 methods)

In [114]:
A = [1 0 0 0; 0 1 0 0]
G = [1 0; 0 1]
M = [1 0; 0 1]

2×2 Matrix{Int64}:
 1  0
 0  1

In [115]:
B = Grad(Pu) ⋅ A + Pu ⋅ G

ChainSum(ChainSumTerm[ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 160801, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs), LowLevelFEM.GradOp()), Any[[1 0 0 0; 0 1 0 0]]), :full), ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 160801, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs), LowLevelFEM.IdOp()), Any[[1 0; 0 1]]), :full)])

In [116]:
B' ⋅ M ⋅ B

CompoundBilinear(TransposedChain(ChainSum(ChainSumTerm[ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 160801, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs), LowLevelFEM.GradOp()), Any[[1 0 0 0; 0 1 0 0]]), :full), ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 160801, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs), LowLevelFEM.IdOp()), Any[[1 0; 0 1]]), :full)])), Any[[1 0; 0 1]], ChainSum(ChainSumTerm[ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_

In [117]:
B = full(A ⋅ Grad(Pu)) + reduced(G ⋅ Pu)

K = ∫(B' ⋅ M ⋅ B; Ω="body", gauss=:auto)

sparse([1, 2, 9, 10, 407, 408, 2799, 2800, 3199, 3200  …  2003, 2004, 82401, 82402, 320801, 320802, 321597, 321598, 321601, 321602], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  321602, 321602, 321602, 321602, 321602, 321602, 321602, 321602, 321602, 321602], [0.6215558641974942, -0.0003333333333333246, -0.03333348765431583, 8.333333333332475e-5, -0.19999969135799212, -0.00016666666666669198, -0.0331668209876577, 0.00011111111111110423, -0.20033302469136688, -0.00044444444444441075  …  0.0017777777777777599, 1.2345679012344108e-6, -0.00022222222222219266, 3.086419753086695e-7, 3.076084851248717e-17, 1.2345679012346005e-6, -0.0017777777777778548, 1.234567901234648e-6, -4.0779554212611036e-17, 4.938271604938119e-6], 321602, 321602)

In [118]:
B1 = A ⋅ Grad(Pu)
B2 = G ⋅ Pu

K2 =
    L.∫(Grad(Pu) ⋅ A' ⋅ M ⋅ A ⋅ Grad(Pu); Ω="body", gauss=:full) +
    L.∫(Grad(Pu) ⋅ A' ⋅ M ⋅ G ⋅ Pu; Ω="body", gauss=:full) +
    L.∫(Pu ⋅ G' ⋅ M ⋅ A ⋅ Grad(Pu); Ω="body", gauss=:full) +
    L.∫(Pu ⋅ G' ⋅ M ⋅ G ⋅ Pu; Ω="body", gauss=:full)

sparse([1, 2, 9, 10, 407, 408, 2799, 2800, 3199, 3200  …  2003, 2004, 82401, 82402, 320801, 320802, 321597, 321598, 321601, 321602], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  321602, 321602, 321602, 321602, 321602, 321602, 321602, 321602, 321602, 321602], [0.6215559999999634, -0.0003333333333333246, -0.03333344444443929, 8.333333333332475e-5, -0.1999997777777452, -0.00016666666666669198, -0.03316677777778115, 0.00011111111111110423, -0.20033311111111995, -0.00044444444444441075  …  0.0017777777777777599, 8.888888888887417e-7, -0.00022222222222219266, 1.1111111111112205e-7, 3.076084851248717e-17, 8.888888888888734e-7, -0.0017777777777778548, 8.888888888890049e-7, -4.0779554212611036e-17, 7.111111111110968e-6], 321602, 321602)

In [119]:
K[:, :]

321602×321602 SparseArrays.SparseMatrixCSC{Float64, Int64} with 10240807 stored entries:
⎡⣿⣿⡛⠛⠛⠛⠛⠛⠛⠛⠛⠿⢿⣟⣛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠋⠉⠉⎤
⎢⣿⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⎥
⎢⣿⡄⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠉⎥
⎢⣿⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⡿⠀⠀⠀⠀⠀⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⡇⠀⠀⠀⠀⠀⠀⠀⠀⠘⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [120]:
K2[:, :]

321602×321602 SparseArrays.SparseMatrixCSC{Float64, Int64} with 10240807 stored entries:
⎡⣿⣿⡛⠛⠛⠛⠛⠛⠛⠛⠛⠿⢿⣟⣛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠋⠉⠉⎤
⎢⣿⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⎥
⎢⣿⡄⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠉⎥
⎢⣿⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⡿⠀⠀⠀⠀⠀⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⡇⠀⠀⠀⠀⠀⠀⠀⠀⠘⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [121]:
K1 = ∫((B1 + B2)' ⋅ M ⋅ (B1 + B2); Ω="body", gauss=:full)

norm(K1.A - K2.A)

0.0

In [122]:
B = A ⋅ Grad(Pu) + G ⋅ Pu

K3 = ∫(B' ⋅ M ⋅ B; Ω="body", gauss=:full)

sparse([1, 2, 9, 10, 407, 408, 2799, 2800, 3199, 3200  …  2003, 2004, 82401, 82402, 320801, 320802, 321597, 321598, 321601, 321602], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  321602, 321602, 321602, 321602, 321602, 321602, 321602, 321602, 321602, 321602], [0.6215559999999634, -0.0003333333333333246, -0.03333344444443929, 8.333333333332475e-5, -0.1999997777777452, -0.00016666666666669198, -0.03316677777778115, 0.00011111111111110423, -0.20033311111111995, -0.00044444444444441075  …  0.0017777777777777599, 8.888888888887417e-7, -0.00022222222222219266, 1.1111111111112205e-7, 3.076084851248717e-17, 8.888888888888734e-7, -0.0017777777777778548, 8.888888888890049e-7, -4.0779554212611036e-17, 7.111111111110968e-6], 321602, 321602)

In [123]:
B = full(A ⋅ Grad(Pu)) + reduced(G ⋅ Pu)

K4 = ∫(B' ⋅ M ⋅ B; Ω="body", gauss=:auto)

sparse([1, 2, 9, 10, 407, 408, 2799, 2800, 3199, 3200  …  2003, 2004, 82401, 82402, 320801, 320802, 321597, 321598, 321601, 321602], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  321602, 321602, 321602, 321602, 321602, 321602, 321602, 321602, 321602, 321602], [0.6215558641974942, -0.0003333333333333246, -0.03333348765431583, 8.333333333332475e-5, -0.19999969135799212, -0.00016666666666669198, -0.0331668209876577, 0.00011111111111110423, -0.20033302469136688, -0.00044444444444441075  …  0.0017777777777777599, 1.2345679012344108e-6, -0.00022222222222219266, 3.086419753086695e-7, 3.076084851248717e-17, 1.2345679012346005e-6, -0.0017777777777778548, 1.234567901234648e-6, -4.0779554212611036e-17, 4.938271604938119e-6], 321602, 321602)

In [124]:
norm(K3.A - K4.A)

0.0008719315451984348

In [125]:
r = ScalarField(Pu, "body", (x, y, z)->x)

elementwise ScalarField
[[1.0; 1.005; … ; 1.0; 1.0025;;], [1.0; 1.005; … ; 1.0; 1.0025;;], [1.0; 1.005; … ; 1.0; 1.0025;;], [1.0; 1.005; … ; 1.0; 1.0025;;], [1.0; 1.005; … ; 1.0; 1.0025;;], [1.0; 1.005; … ; 1.0; 1.0025;;], [1.0; 1.005; … ; 1.0; 1.0025;;], [1.0; 1.005; … ; 1.0; 1.0025;;], [1.0; 1.005; … ; 1.0; 1.0025;;], [1.0; 1.0050000000000001; … ; 1.0; 1.0025;;]  …  [1.995; 2.0; … ; 1.995; 1.9975;;], [1.9949999999999999; 2.0; … ; 1.995; 1.9975;;], [1.995; 2.0; … ; 1.995; 1.9975;;], [1.9949999999999999; 2.0; … ; 1.995; 1.9975;;], [1.995; 2.0; … ; 1.995; 1.9975;;], [1.9949999999999999; 2.0; … ; 1.995; 1.9975;;], [1.995; 2.0; … ; 1.995; 1.9975;;], [1.9949999999999999; 2.0; … ; 1.995; 1.9975;;], [1.995; 2.0; … ; 1.995; 1.9975;;], [1.9949999999999999; 2.0; … ; 1.995; 1.9975;;]]

In [126]:
A1 = [1 0 0; 0 0 0; 0 1 0; 0 0 1]

4×3 Matrix{Int64}:
 1  0  0
 0  0  0
 0  1  0
 0  0  1

In [127]:
A2 = [0 0; 1/r 0; 0 0; 0 0]

4×2 Matrix{Any}:
 0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [128]:
B = A1 ⋅ SymGrad(Pu) + A2 ⋅ Pu

ChainSum(ChainSumTerm[ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 160801, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs), LowLevelFEM.SymGradOp()), Any[[1 0 0 0; 0 0 1 0; 0 0 0 1]]), :full), ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 160801, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs), LowLevelFEM.IdOp()), Any[Any[0 ScalarField([[1.0; 0.9950248756218907; … ; 1.0; 0.9975062344139651;;], [1.0; 0.9950248756218907; … ; 1.0; 0.9975062344139651;;], [1.0; 0.9950248756218907; … ;

In [129]:
E = mat.E
ν = mat.ν

D = E / (1+ν) / (1-2ν) * [1-ν ν ν 0; ν 1-ν ν 0; ν ν 1-ν 0; 0 0 0 (1-2ν)/2]
R = 2π * [r 0 0 0; 0 r 0 0; 0 0 r 0; 0 0 0 r]

4×4 Matrix{Any}:
  ScalarField([[6.28319; 6.3146; … ; 6.28319; 6.29889;;], [6.28319; 6.3146; … ; 6.28319; 6.29889;;], [6.28319; 6.3146; … ; 6.28319; 6.29889;;], [6.28319; 6.3146; … ; 6.28319; 6.29889;;], [6.28319; 6.3146; … ; 6.28319; 6.29889;;], [6.28319; 6.3146; … ; 6.28319; 6.29889;;], [6.28319; 6.3146; … ; 6.28319; 6.29889;;], [6.28319; 6.3146; … ; 6.28319; 6.29889;;], [6.28319; 6.3146; … ; 6.28319; 6.29889;;], [6.28319; 6.3146; … ; 6.28319; 6.29889;;]  …  [12.535; 12.5664; … ; 12.535; 12.5507;;], [12.535; 12.5664; … ; 12.535; 12.5507;;], [12.535; 12.5664; … ; 12.535; 12.5507;;], [12.535; 12.5664; … ; 12.535; 12.5507;;], [12.535; 12.5664; … ; 12.535; 12.5507;;], [12.535; 12.5664; … ; 12.535; 12.5507;;], [12.535; 12.5664; … ; 12.535; 12.5507;;], [12.535; 12.5664; … ; 12.535; 12.5507;;], [12.535; 12.5664; … ; 12.535; 12.5507;;], [12.535; 12.5664; … ; 12.535; 12.5507;;]], Matrix{Float64}(undef, 0, 0), [0.0], [805, 806, 807, 808, 809, 810, 811, 812, 813, 814  …  40795, 40796, 40797, 40

In [130]:
K = ∫(B' ⋅ D ⋅ B, weight=2π*r)
K[:, :]

321602×321602 SparseArrays.SparseMatrixCSC{Float64, Int64} with 10252704 stored entries:
⎡⣿⣿⡛⠛⠛⠛⠛⠛⠛⠛⠛⠿⢿⣟⣛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠋⠉⠉⎤
⎢⣿⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⎥
⎢⣿⡄⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠉⎥
⎢⣿⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⡿⠀⠀⠀⠀⠀⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⡇⠀⠀⠀⠀⠀⠀⠀⠀⠘⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [131]:
K = ∫(B' ⋅ D ⋅ B * (2π*r))
K[:, :]

321602×321602 SparseArrays.SparseMatrixCSC{Float64, Int64} with 10252704 stored entries:
⎡⣿⣿⡛⠛⠛⠛⠛⠛⠛⠛⠛⠿⢿⣟⣛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠋⠉⠉⎤
⎢⣿⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⎥
⎢⣿⡄⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠉⎥
⎢⣿⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⡿⠀⠀⠀⠀⠀⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⡇⠀⠀⠀⠀⠀⠀⠀⠀⠘⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [132]:
R2 = 2π * [r 0; 0 r]

f = L.∫(Pu ⋅ R2 ⋅ [1, 0], Γ="right")

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [133]:
bc = BoundaryCondition("leftbottom", uy=0)

BoundaryCondition("leftbottom", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0))

In [134]:
u = solveField(K, f, support=[bc])

nodal VectorField
[1.3333333281644428e-5; 0.0; … ; 1.3660423446256549e-5; -3.990000084754893e-6;;]

In [135]:
showDoFResults(u, name="u", visible=true, factor=1e5)

0

In [136]:
prob = Problem([mat], type=:AxiSymmetric)

ld = load("right", fx=1)

LoadCondition("right", nothing, Dict{Symbol, Union{Nothing, Function, Number, ScalarField}}(:fy => nothing, :fz => nothing, :fx => 1))

In [137]:
K2 = stiffnessMatrix(prob)
K2[:, :]

321602×321602 SparseArrays.SparseMatrixCSC{Float64, Int64} with 10250996 stored entries:
⎡⣿⣿⡛⠛⠛⠛⠛⠛⠛⠛⠛⠿⢿⣟⣛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠋⠉⠉⎤
⎢⣿⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⎥
⎢⣿⡄⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠉⎥
⎢⣿⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⡿⠀⠀⠀⠀⠀⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⡇⠀⠀⠀⠀⠀⠀⠀⠀⠘⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [138]:
f2 = loadVector(prob, [ld])
u2 = solveDisplacement(K2, f2, support=[bc])

nodal VectorField
[1.333333328161689e-5; 0.0; … ; 1.3660423446367737e-5; -3.9900000847891855e-6;;]

In [139]:
showDoFResults(u2, name="u2", factor=1e5)

1

In [140]:
D

4×4 Matrix{Float64}:
 2.69231e5  1.15385e5  1.15385e5      0.0
 1.15385e5  2.69231e5  1.15385e5      0.0
 1.15385e5  1.15385e5  2.69231e5      0.0
 0.0        0.0        0.0        76923.1

In [141]:
A2

4×2 Matrix{Any}:
 0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [142]:
D * A2

4×2 Matrix{Any}:
 ScalarField([[1.15385e5; 1.14811e5; … ; 1.15385e5; 1.15097e5;;], [1.15385e5; 1.14811e5; … ; 1.15385e5; 1.15097e5;;], [1.15385e5; 1.14811e5; … ; 1.15385e5; 1.15097e5;;], [1.15385e5; 1.14811e5; … ; 1.15385e5; 1.15097e5;;], [1.15385e5; 1.14811e5; … ; 1.15385e5; 1.15097e5;;], [1.15385e5; 1.14811e5; … ; 1.15385e5; 1.15097e5;;], [1.15385e5; 1.14811e5; … ; 1.15385e5; 1.15097e5;;], [1.15385e5; 1.14811e5; … ; 1.15385e5; 1.15097e5;;], [1.15385e5; 1.14811e5; … ; 1.15385e5; 1.15097e5;;], [1.15385e5; 1.14811e5; … ; 1.15385e5; 1.15097e5;;]  …  [57836.9; 57692.3; … ; 57836.9; 57764.5;;], [57836.9; 57692.3; … ; 57836.9; 57764.5;;], [57836.9; 57692.3; … ; 57836.9; 57764.5;;], [57836.9; 57692.3; … ; 57836.9; 57764.5;;], [57836.9; 57692.3; … ; 57836.9; 57764.5;;], [57836.9; 57692.3; … ; 57836.9; 57764.5;;], [57836.9; 57692.3; … ; 57836.9; 57764.5;;], [57836.9; 57692.3; … ; 57836.9; 57764.5;;], [57836.9; 57692.3; … ; 57836.9; 57764.5;;], [57836.9; 57692.3; … ; 57836.9; 57764.5;;]], Matri

In [143]:
openPostProcessor()